# Demo RAG một câu hỏi

Mặc định notebook này **không chạy model** để tránh nặng máy.
Nó đọc kết quả đã có sẵn trong `results/generated_qa_30`.

Nếu muốn chạy thật một câu trên Kaggle, đổi `RUN_HEAVY = True` ở cell cuối.

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
RESULT_DIR = ROOT / "results" / "generated_qa_30"
RAW_PATH = RESULT_DIR / "qwen3_4b__bge_m3.json"

data = json.loads(RAW_PATH.read_text(encoding="utf-8"))
print("Số câu đã có kết quả:", len(data))

Số câu đã có kết quả: 30


In [2]:
question_id = "qa001"
item = next(x for x in data if x["question_id"] == question_id)

print("QUESTION:")
print(item["question"])
print("\nANSWER:")
print(item["answer"])
print("\nLATENCY:", item.get("latency_seconds"), "seconds")

QUESTION:
Cây ăn trái có biểu hiện Hoa khô, rụng hàng loạt, trái non dễ rụng. Nên dùng sản phẩm nào và vì sao?

ANSWER:
1. Chẩn đoán khả năng  
Cây ăn trái có biểu hiện hoa khô, rụng hàng loạt, trái non dễ rụng có thể do thiếu vi lượng Kẽm và Bo trong giai đoạn phân hóa mầm hoa.

2. Sản phẩm gợi ý  
KẼM BORON 50.000ppm

3. Lý do phù hợp  
Sản phẩm KẼM BORON 50.000ppm có thành phần chứa 50.000ppm Kẽm và 50.000ppm Bo, giúp tăng khả năng thụ phấn, đậu trái và hạn chế rụng hoa, trái non – phù hợp với triệu chứng hoa khô, rụng hàng loạt và trái non dễ rụng.

4. Cách dùng tóm tắt  
Pha 500g sản phẩm cho 200–250 lít nước, phun đều hai mặt lá.

5. Lưu ý an toàn  
Không pha chung với thuốc có tính kiềm mạnh. Bảo quản nơi khô ráo.

LATENCY: None seconds


In [3]:
chunks = pd.DataFrame(item["retrieved_chunks"])
display(chunks[["rank", "score", "product_id", "ten_san_pham", "text"]].head(5))

,rank,score,product_id,ten_san_pham,text
0,1,0.704584,sp372,ANTI DROP FLOWER FRUIT,Sản phẩm: ANTI DROP FLOWER FRUIT Loại: thuoc_k...
1,2,0.695642,sp404,NPK 6-20-30 + BO,Sản phẩm: NPK 6-20-30 + BO Loại: phan_bon Rule...
2,3,0.694210,sp306,NPK 8-24-24 + BO,Sản phẩm: NPK 8-24-24 + BO Loại: phan_bon Rule...
3,4,0.683747,sp179,CHỐNG RỤNG HOA TRÁI NAA + BO,Sản phẩm: CHỐNG RỤNG HOA TRÁI NAA + BO Loại: t...
4,5,0.683292,sp1,KẼM BORON 50.000ppm,Sản phẩm: KẼM BORON 50.000ppm Loại: phan_bon R...


In [4]:
# Chỉ bật cell này trên Kaggle khi đã có GPU và muốn chạy thật 1 câu.
RUN_HEAVY = False

if RUN_HEAVY:
    !python3 scripts/benchmark_rag.py \
      --config configs/benchmark_models.yaml \
      --questions data/eval/demo_question.json \
      --top-k 3 \
      --device cuda \
      --llm-device cuda \
      --max-new-tokens 256 \
      --output-dir results/rag_demo_1_question
else:
    print("RUN_HEAVY=False nên không tải model / không chạy GPU.")

RUN_HEAVY=False nên không tải model / không chạy GPU.
